# 6.4 LDA

In [5]:
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
import gensim
import gensim.corpora as corpora

### Load Data

In [6]:
data = pd.read_csv("news_articles.csv") # data should be in the same folder as your notebook

In [7]:
data.head()

,id,title,content
0,25626,"One Weight-Loss Approach Fits All? No, Not Eve...","Dr. Frank Sacks, a professor of nutrition at H..."
1,19551,South Carolina Stuns Baylor to Reach the Round...,South Carolina’s win over Duke was not only ...
2,25221,"U.S. Presidential Race, Apple, Gene Wilder: Yo...",(Want to get this briefing by email? Here’s th...
3,18026,"His Predecessor Gone, Gambia’s New President F...","BANJUL, Gambia — A week after he was inaugu..."
4,21063,‘Harry Potter and the Cursed Child’ Goes From ...,The biggest book of the summer isn’t a blockbu...


In [8]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   id       100 non-null    int64 
 1   title    100 non-null    object
 2   content  100 non-null    object
dtypes: int64(1), object(2)
memory usage: 2.5+ KB


### Clean Data

In [9]:
# take just the content of the article, lowercase and remove punctuation
articles = data['content'].str.lower().apply(lambda x: re.sub(r"([^\w\s])", "", x))

# stop word removal
en_stopwords = stopwords.words('english')
articles = articles.apply(lambda x: ' '.join([word for word in x.split() if word not in (en_stopwords)]))

# tokenize
articles = articles.apply(lambda x: word_tokenize(x))

# stemming (done for speed as we have a lot of text)
ps = PorterStemmer()
articles = articles.apply(lambda tokens: [ps.stem(token) for token in tokens])

In [10]:
articles

0     [dr, frank, sack, professor, nutrit, harvard, ...
1     [south, carolina, win, duke, surpris, fan, pos...
2     [want, get, brief, email, here, good, even, he...
3     [banjul, gambia, week, inaugur, anoth, countri...
4     [biggest, book, summer, isnt, blockbust, thril...
                            ...                        
95    [want, get, brief, email, here, good, even, he...
96    [tallinn, estonia, guard, brought, ahm, abdul,...
97    [gov, scott, walker, wisconsin, activ, wiscons...
98    [social, media, shook, emot, headlin, shout, n...
99    [moment, joanna, acevedo, first, set, foot, bo...
Name: content, Length: 100, dtype: object

### Vectorization

In [11]:
# create dictionary of all words
dictionary = corpora.Dictionary(articles)
print(dictionary)

Dictionary<8693 unique tokens: ['10', '100', '108', '15', '155']...>


In [18]:
# vecotize using bag of words into a document term matrix
doc_term = [dictionary.doc2bow(text) for text in articles]

In [19]:
# Peek at first document: each item is (word_id, count)
print("First article BoW (first 10 pairs):")
print(doc_term[0][:10])
print()

# Map word_id -> actual word so the numbers make sense
print("Same pairs decoded with dictionary:")
for word_id, count in doc_term[0][:10]:
    print(f"  id={word_id:4d}  word={dictionary[word_id]!r:20s}  count={count}")

print()
print(f"Total documents: {len(doc_term)}")
print(f"Unique words in first article (BoW length): {len(doc_term[0])}")


First article BoW (first 10 pairs):
[(0, 1), (1, 1), (2, 1), (3, 2), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (9, 1)]

Same pairs decoded with dictionary:
  id=   0  word='10'                  count=1
  id=   1  word='100'                 count=1
  id=   2  word='108'                 count=1
  id=   3  word='15'                  count=2
  id=   4  word='155'                 count=1
  id=   5  word='180'                 count=1
  id=   6  word='185'                 count=1
  id=   7  word='190'                 count=1
  id=   8  word='1970'                count=1
  id=   9  word='20'                  count=1

Total documents: 100
Unique words in first article (BoW length): 693


In [17]:
# doc_term is a sparse list — no .shape attribute
# Logical dense shape would be: (n_documents, n_unique_words)
n_docs = len(doc_term)
n_vocab = len(dictionary)
print(f"Logical dense shape: ({n_docs}, {n_vocab})  # docs x vocabulary")
print(f"Physical form: list of {n_docs} BoW lists of (word_id, count)")
print(f"Non-zero entries in doc 0: {len(doc_term[0])}")

AttributeError: 'list' object has no attribute 'shape'

### LDA

In [14]:
# specify number of topics
num_topics = 2

In [15]:
# create LDA model
lda_model = gensim.models.LdaModel(corpus=doc_term,
                                   id2word=dictionary,
                                   num_topics=num_topics)

In [16]:
lda_model.print_topics(num_topics=num_topics, num_words=5)

[(0,
  '0.017*"mr" + 0.013*"said" + 0.007*"trump" + 0.004*"would" + 0.004*"year"'),
 (1,
  '0.017*"mr" + 0.017*"said" + 0.005*"state" + 0.005*"would" + 0.005*"one"')]

In [ ]:
### How LDA converges (animation over 20 passes)

Each **pass** = one full sweep through all documents.

On every pass LDA roughly:
1. Infers which topic each word in each document belongs to
2. Updates topic–word distributions and document–topic mixes
3. Bound / perplexity usually improves (lower perplexity = better fit)

Watch the top words per topic shift, and the log-perplexity curve flatten as it converges.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
import numpy as np

n_passes = 20
top_n = 8  # words shown per topic in the animation

# Start with 1 pass, then update for remaining passes so we can snapshot each step
lda_anim = gensim.models.LdaModel(
    corpus=doc_term,
    id2word=dictionary,
    num_topics=num_topics,
    passes=1,
    iterations=50,
    random_state=42,
    eval_every=None,
)

pass_history = []

def snapshot(model, pass_num):
    topics = []
    for t in range(num_topics):
        topics.append(model.show_topic(t, topn=top_n))  # list of (word, prob)
    return {
        "pass": pass_num,
        "log_perplexity": model.log_perplexity(doc_term),
        "topics": topics,
    }

pass_history.append(snapshot(lda_anim, 1))
print(f"Pass 1/{n_passes}  log_perplexity={pass_history[-1]['log_perplexity']:.4f}")

for p in range(2, n_passes + 1):
    lda_anim.update(doc_term)  # another full pass over the corpus
    pass_history.append(snapshot(lda_anim, p))
    print(f"Pass {p}/{n_passes}  log_perplexity={pass_history[-1]['log_perplexity']:.4f}")

print("Done collecting pass history.")

In [ ]:
# Animate: left = top words per topic; right = log-perplexity converging over passes
fig, axes = plt.subplots(1, num_topics + 1, figsize=(5 * (num_topics + 1), 5))
if num_topics == 1:
    axes = [axes]

def draw_frame(frame_idx):
    snap = pass_history[frame_idx]
    for ax in axes:
        ax.clear()

    # Topic word bars
    for t in range(num_topics):
        words, probs = zip(*snap["topics"][t])
        y = np.arange(len(words))
        axes[t].barh(y, probs, color=f"C{t}")
        axes[t].set_yticks(y)
        axes[t].set_yticklabels(words)
        axes[t].invert_yaxis()
        axes[t].set_xlim(0, max(0.05, max(probs) * 1.2))
        axes[t].set_title(f"Topic {t} — pass {snap['pass']}/{n_passes}")
        axes[t].set_xlabel("P(word | topic)")

    # Convergence curve (filled up to current pass)
    ax_conv = axes[num_topics]
    xs = [h["pass"] for h in pass_history[: frame_idx + 1]]
    ys = [h["log_perplexity"] for h in pass_history[: frame_idx + 1]]
    ax_conv.plot(xs, ys, marker="o", color="black")
    ax_conv.set_xlim(1, n_passes)
    all_lp = [h["log_perplexity"] for h in pass_history]
    pad = (max(all_lp) - min(all_lp)) * 0.1 + 1e-6
    ax_conv.set_ylim(min(all_lp) - pad, max(all_lp) + pad)
    ax_conv.set_xlabel("Pass")
    ax_conv.set_ylabel("Log perplexity (lower is better)")
    ax_conv.set_title("Convergence")
    ax_conv.grid(True, alpha=0.3)

    fig.suptitle(
        f"LDA learning — pass {snap['pass']}/{n_passes}  |  log_perplexity={snap['log_perplexity']:.4f}",
        fontsize=13,
    )
    fig.tight_layout()

anim = animation.FuncAnimation(
    fig, draw_frame, frames=len(pass_history), interval=700, repeat=True
)
plt.close(fig)  # prevent static double-display in Jupyter
HTML(anim.to_jshtml())

In [ ]:
# Quick before/after summary (useful even without watching the full animation)
print("=== Pass 1 (start) ===")
for t, words in enumerate(pass_history[0]["topics"]):
    print(f"Topic {t}:", ", ".join(w for w, _ in words))
print(f"log_perplexity: {pass_history[0]['log_perplexity']:.4f}\n")

print(f"=== Pass {n_passes} (after training) ===")
for t, words in enumerate(pass_history[-1]["topics"]):
    print(f"Topic {t}:", ", ".join(w for w, _ in words))
print(f"log_perplexity: {pass_history[-1]['log_perplexity']:.4f}")

# Static convergence plot
lp = [h["log_perplexity"] for h in pass_history]
plt.figure(figsize=(7, 4))
plt.plot(range(1, n_passes + 1), lp, marker="o")
plt.xlabel("Pass")
plt.ylabel("Log perplexity (lower is better)")
plt.title("LDA convergence over 20 passes")
plt.grid(True, alpha=0.3)
plt.show()